# 🇫🇷 Diplomacy Sentiment Tracker — 60 ans de discours politiques français (1958-2022)

## Problématique
**Comment les discours politiques français de 1958 à 2022 traduisent-ils l'évolution des priorités économiques, sociales et diplomatiques du pays à travers le ton, les thématiques et le vocabulaire employé ?**

## Objectif
Étudier l'évolution du langage politique français à travers ~110 000 discours publics afin d'identifier les **thématiques**, les **tonalités** et le **vocabulaire** dominants selon les périodes et les acteurs politiques.

## Plan
| Partie | Contenu |
|---|---|
| 0. Préparation | Chargement, nettoyage, variables utiles (année, présidence, orateur) |
| Axe 1 | Évolution thématique : de quoi parle-t-on à chaque période ? |
| Axe 2 | Ton des discours : positif, neutre ou négatif ? |
| Axe 3 | Vocabulaire : technique ou émotionnel ? quelles ruptures ? |
| Conclusion | Ce qu'il faut retenir, limites et pistes |

> 📁 Le fichier de données `speechs_data.json` doit être placé dans le dossier `data/` (voir le README). Exécutez ensuite tout le notebook (*Restart & Run All*) pour afficher les résultats.

## 0. Préparation des données

### 0.1 Chargement
Le fichier JSON contient une clé `speechs` : une liste de discours, chacun décrit par son titre, ses orateurs, ses mots-clés, ses thématiques, sa date et son texte.

In [ ]:
import json
import ast
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

# Charger le fichier JSON
with open("../data/speechs_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Les discours sont stockés sous la clé "speechs" -> un DataFrame (une ligne = un discours)
df = pd.DataFrame(data["speechs"])

print(df.columns)
df.head()

Index(['title', 'speakers', 'context', 'keywords', 'thematics', 'datetime',
       'readtime', 'kind', 'text'],
      dtype='object')


,title,speakers,context,keywords,thematics,datetime,readtime,kind,text
0,"Interview de M. Jean-Baptiste Djebbari, minist...","[{'name': 'Jean-Baptiste Djebbari', 'function'...",None,"[France - Grande Bretagne, Litige, Zone de pec...","[Economie, Energie - Transports, Politique des...",2021-11-03T12:00:00Z,18 minutes,Texte intégral,DIMITRI PAVLENKO\nBonjour Jean-Baptiste DJEBBA...
1,Conseil des ministres du 3 novembre 2021. Mesu...,[{'name': 'Secrétariat général du Gouvernement...,None,"[Mesure individuelle, Nomination]","[Institutions, Administration - Réforme de l'E...",2021-11-03T12:00:00Z,1 minute,Texte intégral,Le conseil des ministres a adopté les mesures ...
2,Conseil des ministres du 3 novembre 2021. Résu...,"[{'name': 'Barbara Pompili', 'function': 'Mini...",None,"[Communication gouvernementale, Politique du l...","[Société, Ville - Territoires, Logement]",2021-11-03T12:00:00Z,4 minutes,Texte intégral,La ministre de la transition écologique et la ...
3,Conseil des ministres du 3 novembre 2021. Rest...,"[{'name': 'Roselyne Bachelot', 'function': 'Mi...",None,"[Projet de loi, Deuxieme guerre mondiale, Oeuv...","[Institutions, Justice - Droits fondamentaux, ...",2021-11-03T12:00:00Z,3 minutes,Texte intégral,La ministre de la culture a présenté un projet...
4,"Interview de Mme Barbara Pompili, ministre de ...","[{'name': 'Barbara Pompili', 'function': 'Mini...",None,"[Climat, Conferences et sommets internationaux...","[Société, Environnement, Climat]",2021-11-03T12:00:00Z,25 minutes,Texte intégral,MARIE BERNARDEAU\nBonjour Barbara POMPILI.\nBA...


**Description des colonnes**

| Colonne | Contenu |
|---|---|
| `title` | Titre du discours |
| `speakers` | Liste des orateurs (`name` = nom, `function` = fonction) |
| `context` | Contexte (souvent vide) |
| `keywords` | Mots-clés attribués au discours |
| `thematics` | Thématiques (ex. *Société*, *International*, *Economie*) |
| `datetime` | Date du discours (ex. `2021-11-03T12:00:00Z`) |
| `readtime` | Temps de lecture estimé |
| `kind` | Type : *Texte intégral*, *Résumé*… |
| `text` | Texte du discours |

### 0.2 Nettoyage
1. **Valeurs manquantes** : on supprime les discours sans texte.
2. **Doublons** : les discours présents en double (~470 lignes) sont supprimés.
3. **Type de document** : on garde uniquement les **textes intégraux**. Les *résumés* (comptes rendus du Conseil des ministres) ne sont pas des discours, et les lignes *« Télécharger le(s) document(s) »* ne contiennent pas de texte exploitable.
4. **Date** : l'année est tirée de la colonne `datetime`, qui est complète et plus fiable que les dates citées dans les titres.

In [ ]:
# Valeurs manquantes par colonne
print("Valeurs manquantes avant nettoyage :\n", df.isnull().sum())

In [ ]:
nb_initial = len(df)

# 1. Supprimer les discours sans texte
df = df.dropna(subset=["text"])

# 2. Supprimer les doublons exacts (les listes sont converties en texte pour pouvoir être comparées)
df = df[~df.astype(str).duplicated()]

# 3. Garder uniquement les textes intégraux
df = df[df["kind"] == "Texte intégral"]

# Champ "context" vide -> valeur par défaut
df["context"] = df["context"].fillna("Contexte inconnu")

# 4. Date et année à partir de la colonne datetime
df["date"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
df["year"] = df["date"].dt.year
df = df[(df["year"] >= 1958) & (df["year"] <= 2022)]

df = df.reset_index(drop=True)
print(f"{nb_initial} discours au départ -> {len(df)} discours après nettoyage")

### 0.3 Variables utiles
- **`fonction`** : fonction du premier orateur (Président de la République, Premier ministre, ministre…).
- **`orateur_president`** : nom du président **lorsque c'est lui qui parle** (fonction « Président de la République » et nom d'un président de la Ve République, pour ne pas confondre avec un président du Sénat ou d'une commission).
- **`presidence`** : président **en exercice à la date du discours**, quel que soit l'orateur (utile pour comparer les mandats).

In [ ]:
# Les colonnes de listes peuvent être des chaînes si les données viennent d'un CSV
for col in ["speakers", "thematics", "keywords"]:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Fonction du premier orateur
def extract_function(speakers):
    if isinstance(speakers, list) and speakers:
        return speakers[0].get("function") or "Inconnu"
    return "Inconnu"

df["fonction"] = df["speakers"].apply(extract_function)

# Présidents de la Ve République : nom court -> motif à rechercher dans le nom de l'orateur
PRESIDENTS = {
    "De Gaulle": "gaulle", "Pompidou": "pompidou", "Giscard": "giscard",
    "Mitterrand": "mitterrand", "Chirac": "chirac", "Sarkozy": "sarkozy",
    "Hollande": "hollande", "Macron": "macron",
}

def extract_president(speakers):
    """Nom court du président s'il fait partie des orateurs en tant que Président de la République."""
    if isinstance(speakers, list):
        for p in speakers:
            nom = str(p.get("name", "")).lower()
            fonction = str(p.get("function", "")).lower()
            if "président de la république" in fonction:
                for court, motif in PRESIDENTS.items():
                    if motif in nom:
                        return court
    return None

df["orateur_president"] = df["speakers"].apply(extract_president)

# Président en exercice à la date du discours (dates de début de mandat)
debuts_mandat = pd.to_datetime([
    "1959-01-08", "1969-06-20", "1974-05-27", "1981-05-21",
    "1995-05-17", "2007-05-16", "2012-05-15", "2017-05-14",
], utc=True)
noms = list(PRESIDENTS.keys())

def presidence(date):
    if pd.isna(date) or date < debuts_mandat[0]:
        return "Avant 1959"
    return noms[(debuts_mandat <= date).sum() - 1]

df["presidence"] = df["date"].apply(presidence)

print(df["presidence"].value_counts())

### 0.4 Statistiques descriptives

In [ ]:
# Longueur des discours (en caractères)
df["text_length"] = df["text"].str.len()
print(df["text_length"].describe())

# Fonctions les plus fréquentes
print("\nFonctions des orateurs les plus fréquentes :")
print(df["fonction"].value_counts().head(10))

# Nombre de discours par année
df["year"].value_counts().sort_index().plot(kind="bar", figsize=(14, 4), color="steelblue")
plt.title("Nombre de discours par année")
plt.xlabel("Année")
plt.ylabel("Nombre de discours")
plt.tight_layout()
plt.show()

🔎 **Ce qu'il faut retenir** : ~110 000 discours avec texte, d'environ 8 600 caractères en moyenne (médiane 5 200), mais **très inégalement répartis dans le temps** : quelques discours par an avant les années 1980, plus de 2 000 par an à la fin des années 2010.

⚠️ Conséquence importante : **les résultats des premières décennies reposent sur très peu de textes** et doivent être lus avec prudence. On raisonnera en **proportions** (part des discours) plutôt qu'en nombres bruts.

## Axe 1 — Évolution thématique et contextuelle
**Objectif :** comprendre de quoi on parle à chaque période politique.
1. Quels sont les thèmes dominants entre 1958 et 2022 ?
2. Comment les thématiques ont-elles évolué (comparaison Mai 68 / COVID-19) ?
3. Quelle place occupent les relations internationales selon les présidences ?

### 1.1 Quels sont les thèmes dominants ?

In [ ]:
# Une ligne par couple (discours, thématique)
themes = df[["year", "presidence", "thematics"]].explode("thematics").dropna(subset=["thematics"])

# Top 10 des thématiques sur toute la période
top_themes = themes["thematics"].value_counts().head(10)
print(top_themes)

top_themes.sort_values().plot(kind="barh", figsize=(8, 5), color="teal")
plt.title("Thématiques les plus fréquentes (1958-2022)")
plt.xlabel("Nombre de discours")
plt.tight_layout()
plt.show()

In [ ]:
# Thème dominant par décennie, en % des discours de la décennie
themes["decennie"] = (themes["year"] // 10) * 10
nb_discours_decennie = df.groupby((df["year"] // 10) * 10).size()

grands_themes = ["Société", "International", "Institutions", "Economie"]
part = (themes[themes["thematics"].isin(grands_themes)]
        .groupby(["decennie", "thematics"]).size()
        .unstack()
        .div(nb_discours_decennie, axis=0) * 100)

part.plot(marker="o", figsize=(10, 5))
plt.title("Part des discours abordant chaque grand thème, par décennie")
plt.ylabel("% des discours")
plt.xlabel("Décennie")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

🔎 **Interprétation**

Le trio de tête est : **Société** (~32 000 mentions), **International** (~27 000) et **Institutions** (~24 000).

1. **Société** : la parole politique est avant tout tournée vers les questions sociales (emploi, éducation, santé, cohésion). Les dirigeants s'adressent à des publics larges et doivent rassurer, mobiliser ou justifier des réformes.
2. **International** : la France est un acteur global (UE, ONU, OTAN, francophonie). Le président, chef des armées et garant des traités, doit situer le pays face aux crises et aux défis transnationaux.
3. **Institutions** : dans un régime semi-présidentiel, le fonctionnement de l'État (lois, Parlement, décentralisation, réformes constitutionnelles) est un sujet permanent.

**Bilan :** ces trois thèmes forment un triptyque classique : l'État s'adresse à sa *société*, se positionne dans le *monde* et parle de sa *forme* institutionnelle.

Le graphique par décennie montre si ce classement est stable dans le temps ; pour les décennies 1950-1970, il repose sur peu de discours.

### 1.2 Comparaison de deux crises : Mai 68 et COVID-19
On compare la part des discours abordant chaque thématique pendant deux périodes de crise. Les **proportions** permettent de comparer des périodes de tailles très différentes.

In [ ]:
def themes_periode(annees, n=10):
    """Top n des thématiques d'une période, en % des discours de cette période."""
    sous_df = df[df["year"].isin(annees)]
    compte = sous_df["thematics"].explode().value_counts().head(n)
    print(f"Période {annees[0]}-{annees[-1]} : {len(sous_df)} discours")
    return (compte / len(sous_df) * 100).round(1)

mai_68 = themes_periode([1967, 1968, 1969])
covid = themes_periode([2020, 2021])

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, serie, couleur, titre in [(ax[0], mai_68, "orange", "Autour de Mai 68 (1967-1969)"),
                                 (ax[1], covid, "teal", "Crise du COVID-19 (2020-2021)")]:
    if serie.empty:
        a.set_title(titre + " : aucun discours")   # période absente du corpus
        a.axis("off")
        continue
    serie.sort_values().plot(kind="barh", ax=a, color=couleur, title=titre)
    a.set_xlabel("% des discours")
plt.suptitle("Thématiques abordées en période de crise")
plt.tight_layout()
plt.show()

🔎 **Interprétation**
- **COVID-19** : la crise sanitaire fait apparaître des thématiques absentes des autres périodes (*Santé - Protection sociale*, *Epidémie*, *Santé publique*), aux côtés de l'économie. Le contexte de crise oriente fortement les priorités du discours.
- **Mai 68** : le corpus ne compte que **quelques discours** pour 1967-1969 (les thématiques les plus citées n'y apparaissent que 1 à 3 fois). On ne peut donc **pas** en tirer de conclusion solide sur le discours de l'époque ; ce panneau illustre surtout la faible couverture du corpus avant les années 1980.

### 1.3 Quelle place pour les relations internationales selon les présidences ?
Pour chaque présidence, on calcule la **part des discours** dont les thématiques contiennent *International* ou *Relations internationales*.

In [ ]:
THEMES_INTER = {"International", "Relations internationales"}

df["international"] = df["thematics"].apply(
    lambda t: isinstance(t, list) and bool(THEMES_INTER & set(t))
)

ordre = [p for p in PRESIDENTS if p in df["presidence"].unique()]
inter_presidence = (df[df["presidence"].isin(ordre)]
                    .groupby("presidence")
                    .agg(part_internationale=("international", "mean"), nb_discours=("international", "size"))
                    .reindex(ordre))
inter_presidence["part_internationale"] = (inter_presidence["part_internationale"] * 100).round(1)
print(inter_presidence)

ax = inter_presidence["part_internationale"].plot(kind="barh", figsize=(9, 5), color="seagreen")
ax.invert_yaxis()
for i, v in enumerate(inter_presidence["part_internationale"]):
    ax.text(v + 0.5, i, f"{v} %", va="center")
plt.title("Part des discours consacrés à l'international, par présidence")
plt.xlabel("% des discours de la présidence")
plt.tight_layout()
plt.show()

🔎 **Ce qu'il faudra regarder**
- Les présidences les plus récentes (Chirac → Macron) disposent de milliers de discours : leurs pourcentages sont fiables. Les plus anciennes (De Gaulle, Pompidou, Giscard) reposent sur peu de textes (voir la colonne `nb_discours`).
- Hypothèses à vérifier : la part de l'international devrait augmenter lors des moments diplomatiques forts (construction européenne, guerres en Irak et en Libye, interventions au Sahel, COP21, crise ukrainienne).

## Axe 2 — Ton et perception du discours
**Objectif :** mesurer la positivité ou la négativité des discours.
1. Le ton est-il devenu plus positif ou plus négatif au fil des années ?
2. Quels présidents ont le ton le plus positif ?
3. Les périodes de crise font-elles baisser le ton ?
4. Le ton varie-t-il selon la fonction de l'orateur ?

### Méthode
On utilise **TextBlob** avec l'extension française **textblob-fr**, qui donne pour chaque texte :
- une **polarité** de **-1** (très négatif) à **+1** (très positif) ;
- une **subjectivité** de **0** (factuel) à **1** (opinion, émotion).

> ℹ️ L'analyseur français est indispensable : l'analyseur anglais par défaut de TextBlob ne reconnaîtrait presque aucun mot des discours.

> ⏱️ L'analyse de ~95 000 textes peut prendre du temps (plusieurs dizaines de minutes). Pour un essai rapide, travaillez sur un échantillon : `df = df.sample(5000, random_state=42)`.

In [ ]:
from textblob import TextBlob
from textblob_fr import PatternTagger, PatternAnalyzer

tagger, analyzer = PatternTagger(), PatternAnalyzer()

def sentiment_fr(texte):
    """(polarité, subjectivité) d'un texte français."""
    polarite, subjectivite = TextBlob(texte, pos_tagger=tagger, analyzer=analyzer).sentiment
    return pd.Series({"polarite": polarite, "subjectivite": subjectivite})

df[["polarite", "subjectivite"]] = df["text"].apply(sentiment_fr)
df[["polarite", "subjectivite"]].describe()

**Comment lire les scores ?** Les discours officiels restent très mesurés : les moyennes se situent généralement **entre 0 et 0,2**. Plutôt que de chercher des valeurs extrêmes, on compare les groupes **entre eux**. On qualifie le ton ainsi :

| Polarité moyenne | Ton |
|---|---|
| ≥ 0,2 | Positif / optimiste |
| entre -0,2 et 0,2 | **Neutre / mesuré** |
| ≤ -0,2 | Négatif / alarmiste |

In [ ]:
def classer_ton(score):
    if score >= 0.2:
        return "Positif"
    if score <= -0.2:
        return "Négatif"
    return "Neutre / mesuré"

COULEURS_TON = {"Positif": "seagreen", "Neutre / mesuré": "orange", "Négatif": "tomato"}

### 2.1 Le ton a-t-il évolué au fil des années ?

In [ ]:
ton_annee = df.groupby("year").agg(polarite=("polarite", "mean"), nb_discours=("polarite", "size"))

# On ne trace que les années avec au moins 20 discours, pour éviter les moyennes sur 1 ou 2 textes
ton_fiable = ton_annee[ton_annee["nb_discours"] >= 20]

plt.figure(figsize=(12, 5))
plt.plot(ton_fiable.index, ton_fiable["polarite"], marker="o", color="royalblue")
plt.axhline(0, color="grey", linestyle="--")
plt.title("Polarité moyenne des discours par année (années avec au moins 20 discours)")
plt.xlabel("Année")
plt.ylabel("Polarité moyenne")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

🔎 **Ce qu'il faudra regarder**
- La courbe reste-t-elle au-dessus de 0 ? Un ton globalement légèrement positif est attendu dans des discours institutionnels, qui cherchent à rassurer et à mobiliser.
- Les années avec peu de discours sont exclues, car une seule allocution suffit à faire varier fortement la moyenne. Sans ce filtre, les années 1960 (1 à 10 discours par an) produiraient des pics trompeurs.

### 2.2 Quels présidents ont le ton le plus positif ?

In [ ]:
ton_president = (df.dropna(subset=["orateur_president"])
                 .groupby("orateur_president")
                 .agg(polarite=("polarite", "mean"), nb_discours=("polarite", "size"))
                 .sort_values("polarite"))
ton_president["ton"] = ton_president["polarite"].apply(classer_ton)
print(ton_president)

ax = ton_president["polarite"].plot(kind="barh", figsize=(9, 5),
                                    color=ton_president["ton"].map(COULEURS_TON))
for i, (score, n) in enumerate(zip(ton_president["polarite"], ton_president["nb_discours"])):
    ax.text(score, i, f" {score:.2f} ({n} discours)", va="center")
plt.title("Polarité moyenne des discours prononcés par chaque président")
plt.xlabel("Polarité moyenne")
plt.tight_layout()
plt.show()

🔎 **Ce qu'il faudra regarder**
- On ne retient que les discours **prononcés par le président lui-même**. Les présidences anciennes (De Gaulle, Pompidou, Giscard) comptent beaucoup moins de discours : la colonne `nb_discours` permet d'en juger.
- Les écarts entre présidents sont probablement faibles : la parole présidentielle est très codifiée. Un écart de quelques centièmes ne suffit pas à parler de « styles » différents.

### 2.3 Les crises font-elles baisser le ton ?
On compare trois années de crise à la moyenne de toutes les autres années depuis 2000 :
- **2008** : crise financière mondiale ;
- **2015** : attentats (Charlie Hebdo, 13 novembre) ;
- **2020** : crise sanitaire du COVID-19.

In [ ]:
annees_crise = [2008, 2015, 2020]
recent = df[df["year"] >= 2000]

comparaison = pd.Series({
    **{str(a): recent.loc[recent["year"] == a, "polarite"].mean() for a in annees_crise},
    "Autres années\n(2000-2022)": recent.loc[~recent["year"].isin(annees_crise), "polarite"].mean(),
}).round(3)
print(comparaison)

comparaison.plot(kind="bar", rot=0, figsize=(8, 4),
                 color=["tomato", "tomato", "tomato", "grey"])
plt.title("Polarité moyenne : années de crise vs autres années")
plt.ylabel("Polarité moyenne")
plt.tight_layout()
plt.show()

🔎 **Ce qu'il faudra regarder** : si les barres rouges sont nettement **sous** la barre grise, les crises assombrissent le discours. Si elles sont au même niveau, cela signifie qu'en période de crise, les responsables restent **mesurés**, pour rassurer l'opinion tout en appelant à l'effort collectif.

### 2.4 Le ton dépend-il de la fonction de l'orateur ?

In [ ]:
top_fonctions = df["fonction"].value_counts().drop("Inconnu", errors="ignore").head(15).index

ton_fonction = (df[df["fonction"].isin(top_fonctions)]
                .groupby("fonction")["polarite"].mean()
                .sort_values())

ton_fonction.plot(kind="barh", figsize=(11, 7), color=ton_fonction.apply(classer_ton).map(COULEURS_TON))
plt.axvline(df["polarite"].mean(), color="black", linestyle="--", label="moyenne générale")
plt.title("Polarité moyenne des 15 fonctions les plus fréquentes")
plt.xlabel("Polarité moyenne")
plt.legend()
plt.tight_layout()
plt.show()

🔎 **Ce qu'il faudra regarder** : si toutes les fonctions se situent dans la même zone neutre, **la fonction de l'orateur change peu le ton** : le contexte du discours compte alors davantage que le titre de celui qui parle.

## Axe 3 — Langage et transformation du vocabulaire
**Objectif :** observer comment le vocabulaire politique a évolué.
1. Les discours récents sont-ils plus techniques ou plus émotionnels ?
2. Peut-on identifier des ruptures lexicales entre présidences (Mitterrand vs Macron) ?
3. Quels mots-clés émergent après 2000 ?

### 3.1 Vocabulaire plus technique ou plus émotionnel ?
- **Technicité** : nombre de mots techniques (budget, déficit, croissance, réforme…) **pour 1 000 mots**, pour ne pas avantager les textes longs.
- **Charge émotionnelle** : on utilise la **subjectivité** (0 = factuel, 1 = émotionnel). La polarité indique si le ton est positif ou négatif, pas s'il est émotionnel.

In [ ]:
MOTS_TECHNIQUES = r"\b(budgets?|déficits?|croissance|inflation|réformes?|lois?|impôts?|chômage|industries?|marchés?|dette|fiscal\w*)\b"

nb_mots = df["text"].str.split().str.len()
df["technicite"] = df["text"].str.lower().str.count(MOTS_TECHNIQUES) / nb_mots * 1000

df["periode"] = df["year"].apply(lambda a: "Récent (2015-2022)" if a >= 2015 else "Ancien (1958-2014)")
resume = df.groupby("periode")[["technicite", "subjectivite"]].mean().round(3)
print(resume)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
resume["technicite"].plot(kind="bar", ax=ax[0], rot=0, color="steelblue",
                          title="Mots techniques pour 1 000 mots")
resume["subjectivite"].plot(kind="bar", ax=ax[1], rot=0, color="indianred",
                            title="Subjectivité moyenne (0 = factuel, 1 = émotionnel)")
plt.tight_layout()
plt.show()

🔎 **Ce qu'il faudra regarder**
- **Technicité plus forte après 2015 ?** Ce serait cohérent avec la place des questions budgétaires (déficits, dette, réformes) et avec une communication ministérielle de plus en plus « experte ».
- **Subjectivité plus forte ?** Elle pourrait refléter un besoin de mobiliser face aux crises répétées (terrorisme, pandémie, climat) et l'influence des réseaux sociaux.

### 3.2 Ruptures lexicales entre présidences : Mitterrand vs Macron
On compare les mots les plus fréquents dans les discours **prononcés** par chacun. Les mots trop courants (« cette », « plus », « faire »…) sont retirés, sinon ils domineraient les nuages et masqueraient le vocabulaire réellement distinctif.

In [ ]:
import nltk
from wordcloud import WordCloud

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

# Mots vides français + mots très fréquents dans tous les discours politiques
MOTS_VIDES = set(stopwords.words("french")) | {
    "cette", "plus", "faire", "fait", "être", "avoir", "faut", "doit", "peut", "aussi", "tout",
    "tous", "toutes", "bien", "très", "comme", "ainsi", "donc", "alors", "dont", "leurs", "elles",
    "entre", "depuis", "encore", "parce", "quand", "sans", "sous", "avant", "après", "chaque",
    "autres", "autre", "même", "celle", "celui", "ceux", "deux", "trois", "dire", "veux", "c'est",
    "monsieur", "madame", "mesdames", "messieurs", "oui", "non", "là", "ici", "aujourd'hui",
}

def tokens(texte):
    mots = re.findall(r"[a-zàâçéèêëîïôûùüÿœ]+", texte.lower())
    return [m for m in mots if len(m) > 3 and m not in MOTS_VIDES]

def nuage(textes, titre, n=60):
    freq = Counter(m for t in textes for m in tokens(t)).most_common(n)
    wc = WordCloud(width=900, height=450, background_color="white", colormap="tab10")
    plt.figure(figsize=(10, 5))
    plt.imshow(wc.generate_from_frequencies(dict(freq)), interpolation="bilinear")
    plt.axis("off")
    plt.title(titre)
    plt.show()
    return freq

freq_mitterrand = nuage(df.loc[df["orateur_president"] == "Mitterrand", "text"], "Vocabulaire dominant - Mitterrand")
freq_macron = nuage(df.loc[df["orateur_president"] == "Macron", "text"], "Vocabulaire dominant - Macron")

🔎 **Ce qu'il faudra regarder**
- **Mitterrand** : on s'attend à un vocabulaire tourné vers la nation et le collectif (« pays », « société », « monde »), dans un registre institutionnel.
- **Macron** : on s'attend à davantage de termes liés à la transformation et à l'Europe (« projet », « transformation », « européenne »), cohérents avec une rhétorique de la réforme.

Pour mesurer le registre d'action, on peut aussi compter les occurrences de « il faut » et de « nous devons » dans chaque corpus.

### 3.3 Quels mots-clés émergent après 2000 ?
Plutôt que les mots les plus fréquents (toujours les mêmes), on cherche les mots **dont la fréquence a le plus augmenté** entre avant 2000 et après 2000. On suit aussi quelques mots attendus : *numérique*, *climat*, *europe*, *terrorisme*.

In [ ]:
def frequences(textes):
    compte = Counter(m for t in textes for m in tokens(t))
    total = sum(compte.values())
    return pd.Series(compte) / total * 10_000   # fréquence pour 10 000 mots

avant = frequences(df.loc[df["year"] < 2000, "text"])
apres = frequences(df.loc[df["year"] >= 2000, "text"])

# Mots fréquents après 2000 et beaucoup plus fréquents qu'avant
evolution = pd.DataFrame({"avant_2000": avant, "apres_2000": apres}).fillna(0)
evolution = evolution[evolution["apres_2000"] >= 2]          # au moins 2 occurrences pour 10 000 mots
evolution["ratio"] = (evolution["apres_2000"] + 0.1) / (evolution["avant_2000"] + 0.1)
emergents = evolution.sort_values("ratio", ascending=False).head(25)
print(emergents.round(2))

# Suivi de quelques mots-clés dans le temps (part des discours qui les contiennent)
for mot in ["numérique", "climat", "europe", "terroris"]:
    part = df.groupby("year")["text"].apply(lambda s: s.str.lower().str.contains(mot).mean() * 100)
    part[df["year"].value_counts().sort_index() >= 20].plot(label=mot)
plt.title("Part des discours contenant chaque mot-clé")
plt.ylabel("% des discours")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

🔎 **Ce qu'il faudra regarder** : les mots émergents devraient refléter les nouveaux enjeux du XXIᵉ siècle (numérique, climat, transition écologique, terrorisme, pandémie). Comparer les fréquences avant et après 2000 évite de ne retrouver que les mots généraux (« france », « pays »), présents à toutes les époques.

## Conclusion

### Ce que montre déjà le projet
- Le corpus couvre ~110 000 discours, mais il est **très concentré sur les années 2000-2021**.
- Trois thèmes structurent la parole politique : **Société**, **International** et **Institutions**.
- Les crises (COVID-19) font émerger de nouvelles thématiques (santé, épidémie).
- Le ton des discours officiels reste globalement **mesuré** : ni euphorique, ni alarmiste.

### Limites
- **Couverture inégale** : peu de discours avant les années 1980, d'où des comparaisons fragiles entre les premières présidences et les récentes.
- **Analyse de sentiment par dictionnaire** : TextBlob-fr ne comprend ni l'ironie, ni le contexte, ni les tournures diplomatiques.
- **Thématiques attribuées par la source** : on dépend de l'indexation faite par le producteur des données.
- **Interviews** : le texte contient aussi les questions des journalistes, qui influencent les scores.

### Pistes d'amélioration
- Utiliser un modèle de langue français plus récent (ex. CamemBERT) pour le sentiment.
- Isoler les seules répliques de l'orateur dans les interviews.
- Appliquer une modélisation de sujets (LDA, BERTopic) pour découvrir des thèmes sans dépendre de l'indexation.